## 1. Setup & Imports

In [ ]:
# Standard libraries
import os
import sys
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Image processing
import cv2

# Audio processing
import librosa
import librosa.display
import soundfile as sf

# Deep learning
import torch
from torch.utils.data import DataLoader

# Add backend to path for custom data loader
backend_path = Path('../backend/services').resolve()
sys.path.insert(0, str(backend_path))

# Import custom data loader
from data_loader import FER2013Dataset, RAVDESSDataset, create_dataloaders

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Visualization settings
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")
print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"Librosa Version: {librosa.__version__}")

## 2. FER2013 Dataset - Loading & Structure

In [ ]:
# Load FER2013 dataset
print("Loading FER2013 dataset...")
fer2013_train = FER2013Dataset('data/raw/fer2013/train', augment=False)
fer2013_test = FER2013Dataset('data/raw/fer2013/test', augment=False)

# Display dataset information
print(f"\n📊 FER2013 Dataset Statistics:")
print(f"   Training samples: {len(fer2013_train):,}")
print(f"   Test samples: {len(fer2013_test):,}")
print(f"   Total samples: {len(fer2013_train) + len(fer2013_test):,}")
print(f"   Image resolution: 224 × 224 pixels")
print(f"   Input channels: 3 (RGB)")

# Get emotion distribution
train_emotions = [fer2013_train.get_emotion_name(label) for label in fer2013_train.labels]
test_emotions = [fer2013_test.get_emotion_name(label) for label in fer2013_test.labels]

emotion_distribution = Counter(train_emotions)
print(f"\n😊 Emotion Categories (Training):")
for emotion, count in sorted(emotion_distribution.items()):
    percentage = (count / len(train_emotions)) * 100
    print(f"   {emotion:12s}: {count:6,d} samples ({percentage:5.1f}%)")

## 3. FER2013 - Emotion Distribution Analysis

In [ ]:
# Create emotion distribution visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
emotions = list(emotion_distribution.keys())
counts = list(emotion_distribution.values())
colors = plt.cm.Set3(range(len(emotions)))

axes[0].bar(emotions, counts, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_xlabel('Emotion', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
axes[0].set_title('FER2013 Training Set - Emotion Distribution (Bar Chart)', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, (emotion, count) in enumerate(zip(emotions, counts)):
    axes[0].text(i, count + 100, str(count), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=emotions, autopct='%1.1f%%', colors=colors,
             startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'},
             wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})
axes[1].set_title('FER2013 Training Set - Emotion Distribution (Pie Chart)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fer2013_emotion_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Emotion distribution visualization saved!")

## 4. FER2013 - Class Balance Analysis

In [ ]:
# Calculate class balance metrics
counts_array = np.array(counts)
mean_count = counts_array.mean()
std_count = counts_array.std()
cv = (std_count / mean_count) * 100  # Coefficient of variation
imbalance_ratio = counts_array.max() / counts_array.min()  # Max/Min ratio

print("\n⚖️ FER2013 Class Balance Analysis:")
print(f"   Mean samples per class: {mean_count:.0f}")
print(f"   Std deviation: {std_count:.0f}")
print(f"   Coefficient of Variation (CV): {cv:.1f}%")
print(f"   Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}x")
print(f"   Maximum class (happy): {counts_array.max():,} samples")
print(f"   Minimum class (disgust): {counts_array.min():,} samples")
print(f"\n⚠️ Class Imbalance Assessment:")
if imbalance_ratio > 2:
    print(f"   HIGH IMBALANCE DETECTED (ratio: {imbalance_ratio:.2f}x)")
    print(f"   → Recommendation: Use weighted loss functions during training")
    print(f"   → Calculate class weights: weight = 1 / (class_count / total_samples)")
else:
    print(f"   Moderate imbalance (ratio: {imbalance_ratio:.2f}x)")

# Calculate class weights for training
class_weights = torch.FloatTensor([len(train_emotions) / (len(emotion_distribution) * count)
                                   for count in counts])
print(f"\n📊 Calculated Class Weights for Loss Function:")
for emotion, weight in zip(emotions, class_weights):
    print(f"   {emotion:12s}: {weight:.3f}")

## 5. FER2013 - Sample Visualization

In [ ]:
# Visualize sample images from each emotion (7 emotions x 5 samples = 35 images)
fig, axes = plt.subplots(7, 5, figsize=(16, 20))
fig.suptitle('FER2013 Dataset - Sample Images from Each Emotion Class',
             fontsize=16, fontweight='bold', y=0.995)

emotion_indices = {emotion: [] for emotion in emotion_distribution.keys()}
for idx, emotion in enumerate(train_emotions):
    emotion_indices[emotion].append(idx)

for row_idx, emotion in enumerate(sorted(emotion_distribution.keys())):
    indices = emotion_indices[emotion][:5]  # Take first 5 samples

    for col_idx, sample_idx in enumerate(indices):
        ax = axes[row_idx, col_idx]

        sample = fer2013_train[sample_idx]
        image = sample['image'].permute(1, 2, 0).numpy()
        # Denormalize from [-1, 1] or [0, 1] back to [0, 1]
        image = (image - image.min()) / (image.max() - image.min())

        ax.imshow(image)
        if col_idx == 0:
            ax.set_ylabel(emotion.upper(), fontweight='bold', fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])

        # Add border
        for spine in ax.spines.values():
            spine.set_edgecolor('gray')
            spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('figures/fer2013_sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Sample images visualization saved!")

## 6. RAVDESS Dataset - Loading & Structure

In [ ]:
# Load RAVDESS dataset
print("Loading RAVDESS dataset...")
ravdess = RAVDESSDataset('data/raw/ravdess')

# Display dataset information
print(f"\n📊 RAVDESS Dataset Statistics:")
print(f"   Total audio files: {len(ravdess):,}")
print(f"   Audio format: WAV (16-bit PCM)")
print(f"   Sample rate: 16,000 Hz (16 kHz)")
print(f"   Typical duration: ~4-5 seconds per file")
print(f"   Max sequence length: {ravdess.max_length} seconds")
print(f"   Max samples: {ravdess.max_samples:,}")

# Get emotion distribution
ravdess_emotions = [ravdess.get_emotion_name(label) for label in ravdess.emotion_labels]
emotion_dist_ravdess = Counter(ravdess_emotions)

print(f"\n😊 Emotion Categories (RAVDESS):")
for emotion, count in sorted(emotion_dist_ravdess.items()):
    percentage = (count / len(ravdess_emotions)) * 100
    print(f"   {emotion:15s}: {count:4d} samples ({percentage:5.1f}%)")

## 7. RAVDESS - Emotion Distribution Analysis

In [ ]:
# Create emotion distribution visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
emotions_ravdess = list(emotion_dist_ravdess.keys())
counts_ravdess = list(emotion_dist_ravdess.values())
colors_ravdess = plt.cm.Set2(range(len(emotions_ravdess)))

axes[0].bar(emotions_ravdess, counts_ravdess, color=colors_ravdess, edgecolor='black', linewidth=1.5)
axes[0].set_xlabel('Emotion', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
axes[0].set_title('RAVDESS Dataset - Emotion Distribution (Bar Chart)', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, (emotion, count) in enumerate(zip(emotions_ravdess, counts_ravdess)):
    axes[0].text(i, count + 5, str(count), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts_ravdess, labels=emotions_ravdess, autopct='%1.1f%%', colors=colors_ravdess,
             startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'},
             wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})
axes[1].set_title('RAVDESS Dataset - Emotion Distribution (Pie Chart)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/ravdess_emotion_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ RAVDESS emotion distribution visualization saved!")

## 8. RAVDESS - Audio Features Visualization

In [ ]:
# Visualize audio features from sample files (7 emotions x 3 features)
fig, axes = plt.subplots(7, 3, figsize=(16, 18))
fig.suptitle('RAVDESS Dataset - Audio Features Visualization',
             fontsize=16, fontweight='bold', y=0.995)

ravdess_emotion_indices = {emotion: [] for emotion in emotion_dist_ravdess.keys()}
for idx, emotion in enumerate(ravdess_emotions):
    ravdess_emotion_indices[emotion].append(idx)

for row_idx, emotion in enumerate(sorted(emotion_dist_ravdess.keys())):
    indices = ravdess_emotion_indices[emotion]
    if len(indices) > 0:
        sample_idx = indices[0]  # Take first sample of each emotion
        sample = ravdess[sample_idx]

        # Row title
        axes[row_idx, 0].set_ylabel(emotion.upper(), fontweight='bold', fontsize=11, rotation=0, ha='right')

        # Waveform
        waveform = sample['waveform'].numpy()
        axes[row_idx, 0].plot(waveform, color='steelblue', linewidth=0.5)
        axes[row_idx, 0].set_title('Waveform' if row_idx == 0 else '', fontweight='bold')
        axes[row_idx, 0].set_xlim(0, len(waveform))
        axes[row_idx, 0].set_xticks([])
        axes[row_idx, 0].set_yticks([])
        axes[row_idx, 0].grid(alpha=0.3)

        # Mel Spectrogram
        mel_spec = sample['mel_spectrogram'].numpy()
        im1 = axes[row_idx, 1].imshow(mel_spec, aspect='auto', origin='lower', cmap='viridis')
        axes[row_idx, 1].set_title('Mel Spectrogram' if row_idx == 0 else '', fontweight='bold')
        axes[row_idx, 1].set_xticks([])
        axes[row_idx, 1].set_yticks([])

        # MFCC
        mfcc = sample['mfcc'].numpy()
        im2 = axes[row_idx, 2].imshow(mfcc, aspect='auto', origin='lower', cmap='inferno')
        axes[row_idx, 2].set_title('MFCC' if row_idx == 0 else '', fontweight='bold')
        axes[row_idx, 2].set_xticks([])
        axes[row_idx, 2].set_yticks([])

plt.tight_layout()
plt.savefig('figures/ravdess_audio_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ RAVDESS audio features visualization saved!")

## 9. Dataset Statistics & Comparison

In [ ]:
# Create comparison statistics
print("\n" + "="*70)
print("COMPREHENSIVE DATASET STATISTICS")
print("="*70)

# FER2013 statistics
print("\n📷 FER2013 (Facial Expression Dataset):")
print(f"   Total samples: {len(fer2013_train) + len(fer2013_test):,}")
print(f"   Training set: {len(fer2013_train):,} samples (80%)")
print(f"   Test set: {len(fer2013_test):,} samples (20%)")
print(f"   Image dimensions: 224 × 224 × 3 (RGB)")
print(f"   Total pixels per dataset: {(len(fer2013_train) + len(fer2013_test)) * 224 * 224 / 1e6:.1f}M")
print(f"   Estimated storage: ~{(len(fer2013_train) + len(fer2013_test)) * 224 * 224 * 3 / 1e9:.1f} GB (uncompressed)")

# RAVDESS statistics
print("\n🎙️ RAVDESS (Speech Emotion Dataset):")
print(f"   Total samples: {len(ravdess):,}")
print(f"   Audio format: WAV, 16-bit PCM, 16 kHz")
print(f"   Duration per file: ~4 seconds")
print(f"   Total duration: ~{len(ravdess) * 4 / 60:.1f} minutes (~{len(ravdess) * 4 / 3600:.1f} hours)")
print(f"   Max sequence length: {ravdess.max_length} seconds ({ravdess.max_samples:,} samples)")
print(f"   Features extracted:")
print(f"     - Waveform: shape (80,000,)")
print(f"     - MFCC: 13 coefficients × 157 frames")
print(f"     - Mel Spectrogram: 128 bands × variable frames")

# Comparison
print("\n📊 Dataset Comparison:")
comparison_df = pd.DataFrame({
    'Metric': ['Total Samples', 'Modality', 'Emotion Classes', 'Class Balance', 'Data Type'],
    'FER2013': [f'{len(fer2013_train) + len(fer2013_test):,}', 'Visual (Images)', '7', f'Imbalanced (CV: {cv:.1f}%)', 'Image (224×224×3)'],
    'RAVDESS': [f'{len(ravdess):,}', 'Audio (Speech)', '7 (merged to 7)', 'Balanced', 'Audio (16 kHz, WAV)']
})

print(comparison_df.to_string(index=False))
print("\n" + "="*70)

## 10. DataLoader Functionality Test

In [ ]:
# Test DataLoaders
print("Testing DataLoaders...\n")

# Create dataloaders
loaders = create_dataloaders(batch_size=16, num_workers=0)

# Test FER2013 DataLoader
if loaders['fer_train'] is not None:
    print("✓ FER2013 Train DataLoader:")
    batch = next(iter(loaders['fer_train']))
    print(f"   Batch image shape: {batch['image'].shape}")
    print(f"   Batch labels shape: {batch['label'].shape}")
    print(f"   Label dtype: {batch['label'].dtype}")
    print(f"   Sample labels: {batch['label'][:5].tolist()}")
    print(f"   Image value range: [{batch['image'].min():.3f}, {batch['image'].max():.3f}]")

if loaders['fer_test'] is not None:
    print("\n✓ FER2013 Test DataLoader:")
    batch = next(iter(loaders['fer_test']))
    print(f"   Batch image shape: {batch['image'].shape}")
    print(f"   Number of batches: {len(loaders['fer_test'])}")
    print(f"   Total samples: {len(loaders['fer_test'].dataset)}")

# Test RAVDESS DataLoader
if loaders['ravdess'] is not None:
    print("\n✓ RAVDESS DataLoader:")
    batch = next(iter(loaders['ravdess']))
    print(f"   Batch waveform shape: {batch['waveform'].shape}")
    print(f"   Batch MFCC shape: {batch['mfcc'].shape}")
    print(f"   Batch mel_spectrogram shape: {batch['mel_spectrogram'].shape}")
    print(f"   Batch labels shape: {batch['label'].shape}")
    print(f"   Number of batches: {len(loaders['ravdess'])}")
    print(f"   Total samples: {len(loaders['ravdess'].dataset)}")

print("\n✅ All DataLoaders working correctly!")

## 11. Data Quality Assessment & Phase 2 Recommendations

In [ ]:
print("\n" + "="*80)
print("PHASE 1 DATA QUALITY ASSESSMENT & PHASE 2 RECOMMENDATIONS")
print("="*80)

print("\n✅ DATASET STRENGTHS:")
print("\n   FER2013:")
print("     • Large-scale dataset: 35,887 images (extensive training data)")
print("     • Well-structured emotion labels: 7 emotion categories")
print("     • Pre-split train/test sets: 80/20 split for validation")
print("     • Standardized resolution: All images 224×224 pixels")
print("     • Diverse subjects: Various facial poses and expressions")

print("\n   RAVDESS:")
print("     • High-quality professional recordings: Studio environment")
print("     • Clean emotional labeling: Consistent emotion definitions")
print("     • Multiple speakers: 24 professional actors (gender balanced)")
print("     • Standardized audio format: 16-bit PCM, 16 kHz")
print("     • Balanced emotion distribution: Similar samples per class")

print("\n⚠️ IDENTIFIED CHALLENGES:")
print("\n   FER2013:")
print(f"     • Class imbalance: Imbalance ratio {imbalance_ratio:.2f}x (disgust: 436 vs happy: 7,215)")
print("     • Class-wise variation: Coefficient of Variation {:.1f}%".format(cv))
print("     • Potential label noise: Some subjective emotion categorizations")
print("     • Occlusions & variations: Wide range of facial expressions and lighting")

print("\n   RAVDESS:")
print(f"     • Small dataset size: Only {len(ravdess)} samples (limited diversity)")
print("     • Limited speaker variation: Only 24 professional actors")
print("     • English-only speech: May not generalize to other languages")
print("     • Acting data: Acted emotions may differ from natural speech")

print("\n🎯 PHASE 2 MODEL TRAINING RECOMMENDATIONS:")
print("\n   For FER2013 (Facial Emotion Recognition):")
print("     1. Use weighted loss function:")
print("        • torch.nn.CrossEntropyLoss(weight=class_weights)")
print("        • Weights inversely proportional to class frequency")
print("     2. Apply advanced data augmentation:")
print("        • Random horizontal flips, rotations, color jitter")
print("        • CutOut, Mixup, or other augmentation strategies")
print("     3. Use transfer learning:")
print("        • Vision Transformer (ViT): Pre-trained on ImageNet-1K")
print("        • or ResNet-50/152: Proven deep architecture")
print("     4. Implement early stopping:")
print("        • Monitor validation accuracy to prevent overfitting")
print("        • Patience: 10-15 epochs")

print("\n   For RAVDESS (Speech Emotion Recognition):")
print("     1. Use k-fold cross-validation:")
print("        • 5-fold or leave-one-speaker-out (LOSO) validation")
print("        • Prevents overfitting on small dataset")
print("     2. Apply audio-specific augmentation:")
print("        • Time-stretching, pitch-shifting, Gaussian noise")
print("        • SpecAugment (mask time/frequency bins in spectrogram)")
print("     3. Use pre-trained acoustic models:")
print("        • HuBERT: Self-supervised learning from speech")
print("        • or Wav2Vec 2.0: Speech representation learning")
print("     4. Multi-feature fusion:")
print("        • Combine waveform, MFCC, Mel spectrogram, prosodic features")
print("        • Attention mechanisms for feature weighting")

print("\n   Multimodal Fusion (Phase 2 Final Step):")
print("     1. Early fusion: Concatenate features before model")
print("     2. Late fusion: Combine predictions from both modalities")
print("     3. Attention-based fusion: Learn optimal feature combinations")
print("     4. Co-attention mechanisms: Learn cross-modal dependencies")

print("\n" + "="*80)
print("✅ PHASE 1 COMPLETE: Ready for Phase 2 Model Development!")
print("="*80)